# Explaining convolutional networks: Filters, Saliency, GradCAM


What does a CNN do? Based on what does it compute its final classification? This notebook illustrates three methods to dive deeper into these questions:

- **Visualize activations from the first layers:** Understand what aspects of an input image are emphasized in the early stage of a CNN.
- **Saliency Map (vanilla version):** Given a classified image, identify the pixels that most contribute in raising the score for that class.
- **Class Activation Maps (GradCAM):** highlight portions of an image that are most relevant for a particular classification.

These explainability methods are all inherently reliant on the internal structure of a CNN. This is in contrast to other techniques which can be applied to any kind of input/output system and are thus called *model-agnostic*. We'll deal with two perturbation methods:

- **Occlusion:** Effectively removing a part of an image and studying the effecton the output.
- **LIME:** A more sophisticated method to remove different parts synchronously and then study the output :-)



In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
from matplotlib import pyplot as plt

from IPython.display import Image, display
from PIL import Image as PILImage # Import PIL Image explicitly
import ipywidgets as widgets

import tensorflow as tf
import keras
from keras.applications import inception_v3
import keras_hub

backend_name = keras.backend.backend()
print(f"Keras backend: {backend_name}.")
if backend_name == "jax":
    import jax
    print(f"Backend version (JAX): {jax.__version__}")
elif backend_name == "tensorflow":
    import tensorflow as tf
    print(f"Backend version (TensorFlow): {tf.__version__}")
elif backend_name == "torch":
    import torch
    print(f"Backend version (PyTorch): {torch.__version__}")
else:
    print(f"Unknown backend: {backend_name}")


## Auxiliary code

In [ ]:
def make_mosaic(imgs, n_cols):
    b, h, w, c = imgs.shape
    n_rows = int(np.ceil(b / n_cols))

    # pad with black images if needed
    pad = n_rows * n_cols - b
    if pad > 0:
        imgs = np.concatenate(
            [imgs, np.zeros((pad, h, w, c), dtype=imgs.dtype)],
            axis=0
        )

    imgs = imgs.reshape(n_rows, n_cols, h, w, c)
    imgs = imgs.transpose(0, 2, 1, 3, 4)   # (rows, h, cols, w, c)
    mosaic = imgs.reshape(n_rows * h, n_cols * w, c)

    return mosaic

In [ ]:
# @title imports & helperfunctions

# Tensor Manipulation
from skimage.transform import resize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Display
from IPython.display import display as img_display
import matplotlib.cm as cm
from tqdm import tqdm
from tabulate import tabulate

#IO
import requests
import ast


def get_img_array(img_path, size):
    # `img` is a PIL image of size 299x299
    img = keras.preprocessing.image.load_img(img_path, target_size=size)
    # `array` is a float32 Numpy array of shape (size, 3)
    array = keras.preprocessing.image.img_to_array(img)
    # We add a dimension to transform our array into a "batch"
    # of size (1, size, 3)
    array = np.expand_dims(array, axis=0)
    return array


def make_gradcam_heatmap(img_array, model_2d, last_conv_layer_name, pred_index=None):
    # First, we create a model that maps the input image to the activations
    # of the last conv layer as well as the output predictions
    grad_model = tf.keras.models.Model(
        [model_2d.inputs], [model_2d.get_layer(last_conv_layer_name).output, model_2d.output]
    )

    # Then, we compute the gradient of the top predicted class for our input image
    # with respect to the activations of the last conv layer
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    # This is the gradient of the output neuron (top predicted or chosen)
    # with regard to the output feature map of the last conv layer
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # This is a vector where each entry is the mean intensity of the gradient
    # over a specific feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # We multiply each channel in the feature map array
    # by "how important this channel is" with regard to the top predicted class
    # then sum all the channels to obtain the heatmap class activation
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # For visualization purpose, we will also normalize the heatmap between 0 & 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    # resize the heatmap to be the same size as the original image
    heatmap = heatmap.numpy()
    heatmap = resize(heatmap, img_array.shape[1:3])
    heatmap = (tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)).numpy()

    return heatmap


def iter_occlusion(volume, size=4, stride = None):
    # volume: np array in shape 128, 128, 64, 1

    occlusion_center = np.full((size, size, 3), [0], np.float32)

    for y in range(0, volume.shape[0]-size+1, stride):
        for x in range(0, volume.shape[1]-size+1, stride):
            tmp = volume.copy()

            tmp[y:y + size, x:x + size, :] = occlusion_center

            yield x, y, tmp


def rgb_image_occlusion(volume, model_2d, correct_class, occlusion_size, occlusion_stride=None,
                       clip_pred_hm=True):

    print('occluding...')
    if occlusion_stride is None:
        stride = occlusion_size
    elif occlusion_stride > occlusion_size:
        raise ValueError('stride must be smaller or equal size')

    if occlusion_stride == occlusion_size:
        if (not (volume.shape[0] / occlusion_size).is_integer() or
                not (volume.shape[1] / occlusion_size).is_integer()):

            raise ValueError('size does not work with this volume')
    elif occlusion_stride != occlusion_size:
        if (((volume.shape[0] - occlusion_size) % occlusion_stride) != 0 or
                ((volume.shape[1] - occlusion_size) % occlusion_stride) != 0):

            raise ValueError('shape and size do not match')
    heatmap_prob_sum = np.zeros((volume.shape[0], volume.shape[1]), np.float32)
    heatmap_occ_n = np.zeros((volume.shape[0], volume.shape[1]), np.float32)
    total_steps = int(np.prod(((np.array(volume.shape[0:2]) - occlusion_size) / occlusion_stride) + 1))
    with tqdm(total=total_steps, desc="Calculating heatmap") as pbar:
        for n, (x, y, vol_float) in enumerate(iter_occlusion(volume, size=occlusion_size, stride=occlusion_stride)):
            X = vol_float.reshape(1, volume.shape[0], volume.shape[1], volume.shape[2])
            out = model_2d.predict(X, verbose=0)

            heatmap_prob_sum[y:y + occlusion_size, x:x + occlusion_size] += out[0, correct_class]
            heatmap_occ_n[y:y + occlusion_size, x:x + occlusion_size] += 1
            pbar.update(1)
    print("calculating heatmap...")
    heatmap = heatmap_prob_sum / heatmap_occ_n
    if clip_pred_hm:
        cut_off = model_2d.predict(volume.reshape(1, volume.shape[0], volume.shape[1], volume.shape[2]), verbose=0)[0,
                                 correct_class]
        heatmap = np.abs(np.minimum(heatmap - cut_off, 0))
    return heatmap  # , class_pixels


def generate_all_images(img_array, heatmap, modprob):
    f, axs = plt.subplots(1, 3, figsize=(24,8))
    (ax1, ax2, ax3) = axs

    ax1.imshow(img_array[0]/255, vmin = 0, vmax = 1)
    ax2.imshow(img_array[0]/255, vmin = 0, vmax = 1)
    ax2.imshow(heatmap, cmap='jet', vmin = np.min(heatmap), vmax = np.max(heatmap), alpha=0.4)
    im = ax3.imshow(heatmap, cmap='jet', vmin = np.min(heatmap), vmax = np.max(heatmap), alpha=0.4)
    cb = f.colorbar(im, ax=axs.ravel().tolist())
    cb.ax.axhline(y=modprob, linewidth = 3, c='black')


def generate_superimposed_image(img_array, heatmap, save_name = None):
    f, axs = plt.subplots(1, 1, figsize=(5,5))
    (ax1) = axs

    ax1.imshow(img_array[0]/255, vmin = 0, vmax = 1)
    ax1.imshow(heatmap, cmap='jet', vmin = np.min(heatmap), vmax = np.max(heatmap), alpha=0.4)

    plt.axis('off')

    axins = inset_axes(
            ax1,
            width="5%",  # width: 5% of parent_bbox width
            height="100%",  # height: 50%
            loc="lower left",
            bbox_to_anchor=(1.01, 0., 1, 1),
            bbox_transform=ax1.transAxes,
            borderpad=0,
        )
    plt.colorbar(
            matplotlib.cm.ScalarMappable(
                norm=matplotlib.colors.Normalize(vmin=heatmap.min(), vmax=heatmap.max(), clip=False),
                cmap="jet"),
            cax=axins,
            label='',
            ticks=np.trunc(np.linspace(heatmap.min(), heatmap.max(), 5)*100)/100)

    if save_name is not None:
        plt.savefig(save_name, bbox_inches='tight', dpi=300)

def set_model_globals(model_name):
    global model, img_size, preprocess_input, decode_predictions
    if model_name == 'VGG16':
        model_builder = keras.applications.vgg16.VGG16
        img_size = (224, 224)
        preprocess_input = keras.applications.vgg16.preprocess_input
        decode_predictions = keras.applications.vgg16.decode_predictions
    elif model_name == 'ResNet50V2':
        model_builder = keras.applications.resnet_v2.ResNet50V2
        img_size = (224, 224)
        preprocess_input = keras.applications.resnet_v2.preprocess_input
        decode_predictions = keras.applications.resnet_v2.decode_predictions
    elif model_name == 'Xception':
        model_builder = keras.applications.xception.Xception
        img_size = (299, 299)
        preprocess_input = keras.applications.xception.preprocess_input
        decode_predictions = keras.applications.xception.decode_predictions
    else:
        print("Invalid model selected")
        return
    model = model_builder(weights="imagenet")

#widgets
model_dropdown = widgets.Dropdown(
    options=['VGG16', 'ResNet50V2', 'Xception'],
    description='Model:',
)
ok_button = widgets.Button(description="OK")


def on_ok_button_clicked(b):
    set_model_globals(model_dropdown.value)
    if model is not None:
        print(f"Selected Model: {model_dropdown.value}")
        print(f"Image size: {img_size}")
        model.summary()
    else:
        print("No model selected or an error occurred.")

#button link
ok_button.on_click(on_ok_button_clicked)

def load_labels(verbose=0):
  url = 'https://raw.githubusercontent.com/tensorchiefs/dl_course_2024/main/notebooks/imagenet_labels.txt'
  response = requests.get(url)
  if response.status_code == 200:
      content = response.text
      try:
          # Use ast.literal_eval to safely parse the dictionary string
          imagenet_classes = ast.literal_eval(content)
          print("Dictionary loaded successfully!")
          if verbose ==1:
            for key in list(imagenet_classes.keys())[:5]:
                print(f"{key}: {imagenet_classes[key]}")
      except ValueError as e:
          print(f"Error parsing the content: {e}")
  display_options = [f"{index}: {name}" for index, name in imagenet_classes.items()]
  return display_options

display_options = load_labels(verbose=0)
class_dropdown = widgets.Dropdown(options=display_options,description='ImageNet Class:',)

def patch_predict_image(x_range, y_range):
    x1, x2 = x_range
    y1, y2 = y_range
    img_array_patched = np.copy(img_array)
    img_array_patched[:,y1:y2,x1:x2,:]=0
    img_normalized = ((img_array_patched - img_array_patched.min()) / (img_array_patched.max() - img_array_patched.min()) * 255).astype(np.uint8).squeeze()
    plt.imshow(img_normalized, vmin=0, vmax=255),plt.show();

    preds = model.predict(img_array_patched, verbose=0)
    top_5_preds = keras_hub.utils.decode_imagenet_predictions(preds, top=5)[0]
    print(top_5_preds)

# occlusion stride

def calculate_occ_stride(quadratic_size=224):
  occ_pairs=[]
  for occ_size in range(1,quadratic_size):
    for occ_stride in range(1,quadratic_size):
      if occ_stride >= occ_size:
        continue
      if (quadratic_size -occ_size)%occ_stride ==0:
        occ_pairs.append((occ_size,occ_stride))
  occ_pairs=[f'occlusion: {item[0]}, stride: {item[1]}'for item in occ_pairs ]
  return occ_pairs

## Load convnet model "xception"

In this notebook we work with the so called *exception* image classifier. It is a convolutional neural network introduced in late 2016 and represented an evolutionary step in CNNs development (notably the idea of inception module, which we will not discuss in detail ([Paper](https://arxiv.org/abs/1610.02357), [Explainer](https://www.geeksforgeeks.org/computer-vision/xception/)).


We're loading the xception image classifier from *Keras Hub* using a prebuilt identifier. The model comes with all weights as they were trained on the Imagenet image corpus.

In [ ]:
model = keras_hub.models.ImageClassifier.from_preset(
  "xception_41_imagenet",
  # We can configure the final activation of the classifier. Here,
  # we use a softmax activation so our outputs are probabilities.
  activation="softmax",
)

## Load image data

We've made available a set of openly accessible images from the IMAGENET competition. Run the code below to load the data into a numpy zipped storage file. The file will be stored locally. It is then loaded using `np.load`.

In [ ]:
# This downloads the file to the current Colab session
import gdown
file_id = "1YbfxA5xlRYpifmMvrrJexdkNMa6_xx6z"
url = f'https://drive.google.com/uc?id={file_id}'

output = 'xception_image_data.npz' # Rename it to whatever extension you need
gdown.download(url, output, quiet=False)

# Loads example images
example_images = np.load("~/Downloads/xception_image_data.npz")
X = example_images["X"]
y = example_images["y"]

X.shape

In [ ]:
# Plot the first 50 images
mosaic = make_mosaic(X[:50], 10)
PILImage.fromarray(mosaic)

## View activations of shallow layers

The first layers in a convolutional network can be regarded as relatively interpretable image filters. Each layer has several of them, which is why the layer is sometimes also considered to represent a filter bank. The number of filters in the layer is sometimes referred to as the "width" of the CNN (in contrast to the "depth", which is associated with the number of layers the CNN has).

In [ ]:
# We select an image that we use to see what the activations of the channels of the first layer
# show
select_idx = 10
img = X[select_idx:select_idx+1,:,:,:]
print(keras_hub.utils.decode_imagenet_predictions(model.predict(img)))
PILImage.fromarray(img[0])

In [ ]:
act_layer_name = "block1_conv1_act"
#act_layer_name = "block3_sepconv1_act"
#act_layer_name = "block4_sepconv2_act"
#act_layer_name = "block8_sepconv1_act"
#act_layer_name = "block11_sepconv1_act"
#act_layer_name = "block13_sepconv2_act"
#act_layer_name = "block14_sepconv2_act"

layer_names = ["block1_conv1_act", "block3_sepconv1_act", "block4_sepconv2_act"]


In [ ]:
act_layer = model.backbone.get_layer("block1_conv1_act")
shallow_layer_act = keras.Model(model.inputs, act_layer.output)

# Compute activations of the first layer given the image
activations = shallow_layer_act.predict(img)
n_channels = activations.shape[-1]
ncols = np.int16(np.ceil(np.sqrt(n_channels)))
mosaic = make_mosaic(np.transpose(activations, (3, 1, 2, 0)), n_cols = ncols)[:,:,0]
print(mosaic.shape)
fig, ax = plt.subplots(figsize=(10,10))
vmax = np.percentile(mosaic.ravel(),99)
ax.imshow(mosaic, vmax=vmax)

In [ ]:
act_layer = model.backbone.get_layer("block3_sepconv1_act")
shallow_layer_act = keras.Model(model.inputs, act_layer.output)

# Compute activations of the first layer given the image
activations = shallow_layer_act.predict(img)
n_channels = activations.shape[-1]
ncols = np.int16(np.ceil(np.sqrt(n_channels)))
mosaic = make_mosaic(np.transpose(activations, (3, 1, 2, 0)), n_cols = ncols)[:,:,0]
print(mosaic.shape)
fig, ax = plt.subplots(figsize=(10,10))
vmax = np.percentile(mosaic.ravel(),99)
ax.imshow(mosaic, vmax=vmax)

In [ ]:
# block8_sepconv1_act
act_layer = model.backbone.get_layer("block11_sepconv1_act")
shallow_layer_act = keras.Model(model.inputs, act_layer.output)

# Compute activations of the first layer given the image
activations = shallow_layer_act.predict(img)
n_channels = activations.shape[-1]
ncols = np.int16(np.ceil(np.sqrt(n_channels)))
mosaic = make_mosaic(np.transpose(activations, (3, 1, 2, 0)), n_cols = ncols)[:,:,0]
mosaic.shape
fig, ax = plt.subplots(figsize=(10,10))
vmax = np.percentile(mosaic.ravel(),99)
ax.imshow(mosaic, vmax = vmax)

In [ ]:
act_layer = model.backbone.get_layer("block13_sepconv2_act")
shallow_layer_act = keras.Model(model.inputs, act_layer.output)

# Compute activations of the first layer given the image
activations = shallow_layer_act.predict(img)
n_channels = activations.shape[-1]
ncols = np.int16(np.ceil(np.sqrt(n_channels)))
mosaic = make_mosaic(np.transpose(activations, (3, 1, 2, 0)), n_cols = ncols)[:,:,0]
mosaic.shape
fig, ax = plt.subplots(figsize=(10,10))
vmax = np.percentile(mosaic.ravel(),99)
ax.imshow(mosaic, vmax = vmax)

### ✏️ **YOUR OBSERVATIONS:**

Each mosaic above shows you the activation maps of all channels in a layer of the xception network. The first mosaic comes from the front most batch of layers, the second mosaic comes from the layers further down (deeper) into the network, and so on.

What do you observe?

<details>
  <summary>🔑 Click here to View Answers:</summary>

- 1st Layer: Focus on low-level features such as edges, color-changes, 2D frequencies, smoothing.
- 2nd layer: Still relatively low-level features, there seems to be a stronger focus on edges.
- Deeper layers: Things get hard to decipher relatively quickly. Note that the deeper levels have many more channels.

</details>



### ✏️ **INCREASING CHANNEL NUMBERS:**

As we get deeper into the CNN, the size of the activation maps decreases (size of the tiles), while the number of channels (number of tiles) increases. Do you remember why the number of channels typically increases as we get deeper into a CNN?

<details>
  <summary>🔑 Click here to View Answers:</summary>

The size of the activation maps corresponds to spatial resolution, while the number of the channels corresponds to a type of semantic vocabulary.

As spatial resolution (and information) is reduced, the semantic information is increased. Here's a good intuition from Google Gemini:

- Layer 1: Detects basic edges (e.g., 32 channels).

- Layer 2: Needs to detect combinations of those edges (e.g., "horizontal" + "vertical" = "corner").

- Later Layers: There are many more ways to combine corners into shapes (squares, triangles, eyes) than there are ways to make a corner from an edge. Therefore, you need more channels to hold all those possible higher-level combinations.

</details>



## Occlusion

We inspect which regions of the image contribute to the likelihood of an instance belonging to a class by occluding a part of the image.

Move the sliders below to occlude a part of the input image and see how the predictions change.

-  Can you find a spot which is marginally influencing the prediction?
-  or a spot where the likelihood is getting even higher for elephant??
-  what happens if create a large vertical patch ?


In [ ]:
x_slider = widgets.IntRangeSlider(value=[100, 180],min=0,max=img_array.shape[1]-1,step=1,description='X Range:',continuous_update=False)
y_slider = widgets.IntRangeSlider(value=[100, 180],min=0,max=img_array.shape[1]-1,step=1,description='Y Range:',continuous_update=False)
widgets.interact(patch_predict_image, x_range=x_slider, y_range=y_slider);

## "Vanilla" Saliency map

When we train neural networks, we provide them with an input image and ask them to move their output as much as possible on the neuron corresponding to the correct target value. Technically, this is done through gradient descent on the model weights.

- Given the input, the
- gradient over all model weights is computed
- w.r.t. the loss on the target.

The reverse can be done if we want to understand what parts of an input image are important for a given model, with given weights, to reach its given output.

- Given the output,
- the gradient over all input pixels is computed
- w.r.t. prediction score on a particular output neuron.

This is called **gradient ascent**. The code cell below computes the saliency map for an particular image chosen. Run the code first and look at the saliency map.

In [ ]:
# This function receives an input image, a neural network model, and optionally
# a target class index. It performs gradient ascent on the input image w.r.t. the
# score that the model produces for the class index.
# If no class index is provided, it defaults to the class index that the image
# maximally activates.
def generate_saliency_map(input_image_array, model, class_index=None):

    # Ensure the input image array is float32 and trackable by GradientTape
    # Note that `input_tensor` needs to be made a *trainable* variable, i.e.
    # an object for whose elements a gradient needs to be computed.
    input_tensor = tf.cast(input_image_array, tf.float32)
    input_tensor = tf.Variable(input_tensor, trainable=True)

    # Within the `GradientTape` scope all (implied) computations are tracked for
    # gradient building
    # Effectively, these computations are: From an input image compute the output
    # Class score
    with tf.GradientTape() as tape:
        # Preprocess the input image for the model
        preprocessed_input = model.preprocessor(input_tensor)
        # Get model predictions
        predictions = model(preprocessed_input)
        # NOTE: You can't call `model.predict(input_tensor)` for technical reasons
        #  concerning how `predict` is optimized and how its computations are
        #  not meant to be tracked for gradient calculations.

        if class_index is None:
            # Get the index of the top predicted class
            top_pred_index = tf.argmax(predictions[0])
        else:
            top_pred_index = class_index

        # Get the score for the top predicted class
        top_class_output = predictions[:, top_pred_index]

    # Compute gradients of the top class output with respect to the input image
    gradients = tape.gradient(top_class_output, input_tensor)

    # Take the absolute value of the gradients to highlight positive and negative influences
    # Then, sum across the color channels to get a single saliency map
    saliency_map = tf.reduce_max(tf.abs(gradients), axis=-1)

    # Normalize the saliency map to range [0, 1]
    saliency_map = saliency_map[0].numpy()
    saliency_map /= np.max(saliency_map)

    return saliency_map

# Get the image to analyze (using the same image as for GradCAM)
# Make sure to use the original, un-preprocessed image for saliency calculation
input_image_for_saliency = X[10:11, :, :, :]

# Generate the saliency map
saliency_map = generate_saliency_map(input_image_for_saliency, model)

# Display the saliency map
plt.figure(figsize=(8, 8))
plt.imshow(input_image_for_saliency[0])
plt.imshow(saliency_map, cmap='jet', alpha=0.5) # Overlay with some transparency
plt.title('Saliency Map Overlay')
plt.axis('off')
plt.show()

# You can also display just the saliency map
plt.figure(figsize=(8, 8))
plt.imshow(saliency_map, cmap='gray')
plt.title('Raw Saliency Map')
plt.axis('off')
plt.show()

### ✏️ **YOUR TASK:**

What are your observations? Do you find the saliency map useful?


<details>
  <summary>🔑 Click here to View Answers:</summary>


- Foremost: The saliency map is noisy.
- It's not clear why various areas with activated pixels are important to the classification. This is, by the way, related to the phenomenon of adversarial images (see lecture).
- However, there does seem to be a somewhat loose connection between the activated pixels and the chain saw (the predicted target class).

</details>



### ⚙️ **STUDY THE CODE**

Return to the code above and try to go step by step through the function `generate_saliency_map`: Do you see how the gradient is computed?

Try creating saliency maps for different images.


<details>
  <summary>🔑 Click here to View Answers:</summary>


- Foremost: The saliency map is noisy.
- It's not clear why various areas with activated pixels are important to the classification. This is, by the way, related to the phenomenon of adversarial images (see lecture).
- However, there does seem to be a somewhat loose connection between the activated pixels and the chain saw (the predicted target class).

</details>



## Gradient-weighted Class Activation Mapping (GradCAM)

### Load image

We will now load a test image to perform a classification on and create a heatmap telling us which parts of the image contributed most to the top classification.

In [ ]:
# Access file in google drive mount
# (only required when running on personal colab, without github link)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Downloads the image and stores it locally under the path img_path
img_path = keras.utils.get_file(
    fname="elephant.jpg",
    origin="https://img-datasets.s3.amazonaws.com/elephant.jpg",
)
# Returns a Python Imaging Library (PIL) image
img = keras.utils.load_img(img_path)
img_array = np.expand_dims(img, axis=0)

In [ ]:
img = keras.utils.load_img("drive/MyDrive/Colab Notebooks/n02981792_catamaran.JPEG")
img_array = np.expand_dims(img, axis=0)

In [ ]:
#150
img_array = X[10:11,:,:,:]

In [ ]:
# Compute a 1000 element vector of soft predictions
preds = model.predict(img_array)
print(f"Shape of the output: {preds.shape}")
print(f"Index of maximum probability: {np.argmax(preds[0])}")

# This convenience function will translate the predicted vector into something more human-readable
print(keras_hub.utils.decode_imagenet_predictions(preds))

PILImage.fromarray(img_array[0])

### Prepare two models

The procedure to produce a Gradient Class Activation Map goes as follows:

**Step 1:** Extract a submodel that takes the image input and outputs the activation maps of the last layer before the final classification head of the CNN starts. We call that `submodel_act`.

- Input: a n x m x 3 image array.
- Output: 2048 activation maps of size 10 x 10 each (so a 10 x 10 x 2048 array).

**Step 2:** Create a second model that takes the activations and produces the final classification. We call that `submodel_class`.

- Input: 10 x 10 x 2048 activation map.
- Output: 1000-element vector of soft classification scores over the 1000 target classes.

**Step 3:** Compute the activation maps for the input image using submodel 1. We call these activation maps `act_maps` (shape: `(10,10,2048)`).

**Step 4:** Then take submodel 2 (the one connecting `act_maps` to a 1000-class classification output) and compute its gradient on the actual target class with respect to the input activation maps (those 10x10x2048 numbers). We call that gradient `G` with shape `(10,10,2048)`.

**Step 5:** Compute the average gradient in each of the 2048 panels. This results in a weight vector of size 2048.

**Step 5:** Weight each of the 10x10 panels in `act_maps` (there are 2048 of them) with the 2048 weight vector calculated in step 5.

**Step 6:** Average over all those 2048 activation maps and overlay the resulting 10x10 panel on the image.


In [ ]:
# For reference, look at the high-level summary of the xception network
model.summary()

In [ ]:
# It's really the `xception_backbone` part that has most of the computation in the
# xception network. We can also look at its summary, but mind you: It's going to be a
# loooong chain of elements, connected in ways that we have not covered in class yet.
#
# What we're really after, though, is the final layer calle `block14_sepconv2_act`.
# That layer output those activation maps we mentioned above.
model.backbone.summary()

In [ ]:
# Forward model: This model takes as input an Image and outputs the
# activations of all channels of the last convolutional layer
# These will have substantially lower spatial dimensions (because of information
# compression).
last_conv_layer_name = "block14_sepconv2_act"
last_conv_layer = model.backbone.get_layer(last_conv_layer_name)
submodel_act = keras.Model(model.inputs, last_conv_layer.output)

In [ ]:
# Let's test that forward model: We feed it the image array and look at what comes out

# The xception model has a preprocessor that is run by the `.predict()` method
# but is not included when calling `model.input`. We therefore have to manually
# preprocess the image array
img_array = model.preprocessor(img_array)

output_of_last_layer = submodel_act.predict(img_array)
print(f"The shape of the output of the last layer: {output_of_last_layer.shape}")

What do you make of the shape of the last layer's output? Do you understand the size of the response? What does every index point to?

In [ ]:
# Classification sub-model
# The input are the activations of the last layer, the output is the
# classification, i.e. it applies the gloval average pooling (plus dropout)
# which will condense the spatial activations of all channels into single numbers
# and a dense layer which turns those numbers (one per channel) into a prediction.
classifier_input = last_conv_layer.output
x = classifier_input
for layer_name in ["pooler", "predictions"]:
    x = model.get_layer(layer_name)(x)
submodel_class = keras.Model(classifier_input, x)


In [ ]:
# We run the output activations of the last layer into the classification
# sub-model and confirm that the classification is as expected
keras_hub.utils.decode_imagenet_predictions(
    submodel_class.predict(output_of_last_layer)
    )

### Compute the class activation gradient

The snippet below performs a computation within a `GradienTape` Python context: It takes the activations (outputs of submodel 1 above) and then compute the 1000 class probabilities (submodel 2). It then takes the class probability of with the maximum value (i.e. the activation that points to the predicted class).

Tensorflow will watch the computations necessary to compute the predictions (that's the the tape-context tells it to do) such that it can then compute the gradients using backpropagation. How can we interpret that gradient? It says in which direction (positiv/negative) and by how much we'd have to change the 10x10x2048 activations in order to maximally affect the predicted probability of the target class.

In [ ]:
from keras import ops
import tensorflow as tf

def get_top_class_gradients(img_array):
    # Computes activations of the last conv layer and makes the tape
    # watch it
    last_conv_layer_output = submodel_act(img_array)
    with tf.GradientTape() as tape:
        tape.watch(last_conv_layer_output)
        preds = submodel_class(last_conv_layer_output)
        top_pred_index = ops.argmax(preds[0])
        # Retrieves the activation channel corresponding to the top
        # predicted class
        top_class_channel = preds[:, top_pred_index]

    # Gets the gradient of the top predicted class with respect to the
    # output feature map of the last convolutional layer
    grads = tape.gradient(top_class_channel, last_conv_layer_output)
    return grads, last_conv_layer_output

grads, last_conv_layer_output = get_top_class_gradients(img_array)

grads = ops.convert_to_numpy(grads)
last_conv_layer_output = ops.convert_to_numpy(last_conv_layer_output)

In [ ]:
grads.shape

In [ ]:
# This is a vector where each entry is the mean intensity of the
# gradient for a given channel. It quantifies the importance of each
# channel with regard to the top predicted class.
pooled_grads = np.mean(grads, axis=(0, 1, 2))
last_conv_layer_output = last_conv_layer_output[0].copy()


In [ ]:
last_conv_layer_output.shape

In [ ]:
# Multiplies each channel in the output of the last convolutional layer
# by how important this channel is
for i in range(pooled_grads.shape[-1]):
    last_conv_layer_output[:, :, i] *= pooled_grads[i]

# The channel-wise mean of the resulting feature map is our heatmap of
# class activation
heatmap = np.mean(last_conv_layer_output, axis=-1)

In [ ]:
heatmap = np.maximum(heatmap, 0)
heatmap /= np.max(heatmap)
plt.matshow(heatmap)


In [ ]:
import matplotlib.cm as cm

# Convert the image data to numpy array
#img = keras.utils.img_to_array(img)
img = img_array[0]*255

# Rescale the heatmap to the range 0–255
heatmap = np.uint8(255 * heatmap)

# Use the "jet" colormap to recolorize the heatmap
jet = cm.get_cmap("jet")
jet_colors = jet(np.arange(256))[:, :3]  # Drop the alpha channel (column 3)
jet_heatmap = jet_colors[heatmap]

# Creates an image that contains the recolorized heatmap
jet_heatmap = keras.utils.array_to_img(jet_heatmap)
jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
jet_heatmap = keras.utils.img_to_array(jet_heatmap)

# Superimposes the heatmap and the original image, with the heatmap at
# 40% opacity
superimposed_img = jet_heatmap * 0.5 + img
superimposed_img = keras.utils.array_to_img(superimposed_img)

# Shows the superimposed image
plt.imshow(superimposed_img)